In [3]:
from openai import OpenAI

In [4]:
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

In [5]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': 'hai',
        }
    ],
    model='llama3.1',
)
print(chat_completion.choices[0].message.content)

Hai! Siap membantu apa yang kamu butuhkan hari ini?


In [8]:
from my_agent.utils import OpenAIGPT

In [9]:
from dotenv import load_dotenv
load_dotenv()

llm = OpenAIGPT()

In [10]:
from langchain_core.messages import HumanMessage
for token in llm.stream("hai"):
    print(token.content, end='', flush=True)

Hai!apa yang bisa saya bantu hari ini?

In [13]:
from typing import TypedDict
from pydantic import BaseModel
class state(BaseModel):
    answer: str
    justification: str

In [14]:
state.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'justification': {'title': 'Justification', 'type': 'string'}},
 'required': ['answer', 'justification'],
 'title': 'state',
 'type': 'object'}

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool
convert_to_openai_tool(state)

{'type': 'function',
 'function': {'name': 'state',
  'description': '',
  'parameters': {'properties': {'answer': {'type': 'string'},
    'justification': {'type': 'string'}},
   'required': ['answer', 'justification'],
   'type': 'object'}}}

In [16]:
struct_llm = llm.with_structured_output(state)

In [17]:
class State(BaseModel):
    vajid: str
    good: str

In [18]:
struct_llm.invoke('which coutry have done most wars')

AIMessage(content='{"answer":"United States of America","justification":"According to various sources, the United States has involved itself in numerous conflicts and wars throughout its history, including World War I, World War II, Korea, Vietnam, Gulf Wars, Afghanistan, among others. This makes it one of the most intervened countries globally."}', additional_kwargs={'parsed': state(answer='United States of America', justification='According to various sources, the United States has involved itself in numerous conflicts and wars throughout its history, including World War I, World War II, Korea, Vietnam, Gulf Wars, Afghanistan, among others. This makes it one of the most intervened countries globally.')}, response_metadata={}, id='lc_run--019ea7c2-1005-7063-99e6-b641659d43fd-0', tool_calls=[], invalid_tool_calls=[])

In [19]:
from langchain_core.output_parsers import JsonOutputParser
text = """so tha is the import the {"answer": "and output"}"""
parser = JsonOutputParser()
parser.invoke('{"hai": "hallow"}')

{'hai': 'hallow'}

In [20]:
from langchain_core.output_parsers import PydanticOutputParser

In [21]:
parser = PydanticOutputParser(pydantic_object=state)

In [22]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"title": "Answer", "type": "string"}, "justification": {"title": "Justification", "type": "string"}}, "required": ["answer", "justification"]}
```


In [84]:
from langchain_core.tools import tool
from langchain_core.messages import ToolCall, ToolMessage, AIMessage

@tool
def calculator(expression: str):
    """Calculate any Expression like '2 + 5' or 2 / 6 etc."""
    return eval(expression)
tool_llm = llm.bind_tools([calculator])

user_input = input("Ask a arithematic question")

messages = [
    HumanMessage(content=user_input)
]
out = tool_llm.invoke(messages)
print(out.tool_calls)

tool_out = calculator.invoke(out.tool_calls[0]['args'])
print(tool_out)


messages.append(ToolMessage(content=f'{tool_out}',tool_call_id=out.tool_calls[0]['id']))
print("="*30)
print(llm.invoke(messages).content)

[{'name': 'calculator', 'args': {'expression': '7/1'}, 'id': '7dbb27d1-6406-47b3-af73-00511e6f0d42', 'type': 'tool_call'}]
7.0
The result of the division is a floating-point number because a denominator of 1 effectively divides any number by itself, resulting in a value near 1. 

If you want an integer result, you can use the ceiling function, which rounds up to the nearest whole number.

However, most of the time, you'll likely need this type of operation as a float or decimal for mathematical operations:

```
7 / 1 = 7.0
```


In [81]:
from langchain_core.messages import convert_to_openai_messages
convert_to_openai_messages(messages)

[{'role': 'user',
  'content': 'what is 5th multiply of 10 and multiply it with 400 and subtract 3 from it'},
 {'role': 'tool',
  'tool_call_id': '3c49f1c4-25d3-48c6-808b-bf523c9dca9f',
  'content': '19997'}]

In [15]:
import ollama

def get_weather(city: str):
    return f"The weather in {city} is 22°C and sunny."

def execute(name, args):
    if name == "get_weather":
        return get_weather(**args)

messages = [{"role": "user", "content": "What is the weather in Tokyo?"}]

response = ollama.chat(
    model="llama3.1",
    messages=messages,
    tools=[{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name"
                    }
                },
                "required": ["city"]
            }
        }
    }],
    stream=True,
)


In [18]:
import ollama
import json

def get_weather(city: str):
    return f"The weather in {city} is 22°C and sunny."

def execute(name, args):
    if name == "get_weather":
        return get_weather(**args)

messages = [{"role": "user", "content": "What is the weather in Tokyo?"}]

response = ollama.chat(
    model="llama3.1",
    messages=messages,
    stream=True,
    tools=[{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"}
                },
                "required": ["city"]
            }
        }
    }]
)

tool_call_detected = False

for chunk in response:
    # As soon as we detect a tool call starting, stop printing
    if chunk.message.tool_calls:
        tool_call_detected = True

    if not tool_call_detected and chunk.message.content:
        print(chunk.message.content, end="", flush=True)

    if chunk.done:
        if tool_call_detected:
            for tool in chunk.message.tool_calls:
                result = execute(tool.function.name, tool.function.arguments)

            messages.append({
                "role": "assistant",
                "content": "",
                "tool_calls": [{"function": {"name": t.function.name, "arguments": t.function.arguments}} for t in chunk.message.tool_calls]
            })
            messages.append({"role": "tool", "content": str(result)})

            # Stream final answer
            for final_chunk in ollama.chat(model="llama3.1", messages=messages, stream=True):
                print(final_chunk.message.content or "", end="", flush=True)

TypeError: 'NoneType' object is not iterable

In [ ]:
response.content


AttributeError: 'generator' object has no attribute 'content'

In [16]:
import ollama
import json

MODEL = "qwen3.5"

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country, e.g. 'London, UK'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical calculation",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '2 + 2'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

def execute_tool(name, args):
    if name == "get_weather":
        location = args.get("location", "Unknown")
        unit = args.get("unit", "celsius")
        return {
            "location": location,
            "temperature": 22 if unit == "celsius" else 71,
            "unit": unit,
            "condition": "Sunny",
            "humidity": "60%"
        }
    elif name == "calculate":
        expr = args.get("expression", "")
        try:
            return {"expression": expr, "result": eval(expr)}
        except Exception as e:
            return {"error": str(e)}
    return {"error": "Unknown tool"}


def chat_with_tools(user_message):
    print(f"\n{'='*50}")
    print(f"User: {user_message}")
    print(f"{'='*50}")

    messages = [{"role": "user", "content": user_message}]

    # --- Round 1: Stream first response ---
    print("\nAssistant: ", end="", flush=True)

    collected_content = ""
    collected_tool_calls = []

    stream = ollama.chat(
        model=MODEL,
        messages=messages,
        tools=tools,
        stream=True
    )

    for chunk in stream:
        msg = chunk.message

        # thinking
        if msg.thinking:
            print(msg.thinking, end="", flush=True)

        # Stream text content
        if msg.content:
            print(msg.content, end="", flush=True)
            collected_content += msg.content

        # Collect tool calls (arrive as complete objects, not streamed token by token)
        if msg.tool_calls:
            for tc in msg.tool_calls:
                collected_tool_calls.append(tc)

    print()  # newline after streamed content

    # No tool calls — we're done
    if not collected_tool_calls:
        return collected_content

    # --- Round 2: Execute tools ---
    messages.append({
        "role": "assistant",
        "content": collected_content or "",
        "tool_calls": [
            {
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            }
            for tc in collected_tool_calls
        ]
    })

    for tc in collected_tool_calls:
        name = tc.function.name
        args = tc.function.arguments

        print(f"\n[Tool Called] {name}({json.dumps(args)})")
        result = execute_tool(name, args)
        print(f"[Tool Result] {json.dumps(result)}")

        messages.append({
            "role": "tool",
            "content": json.dumps(result)
        })

    # --- Round 3: Stream final response ---
    print("\nAssistant: ", end="", flush=True)

    final_content = ""

    final_stream = ollama.chat(
        model=MODEL,
        messages=messages,
        tools=tools,
        stream=True
    )

    for chunk in final_stream:
        msg = chunk.message
        if msg.content:
            print(msg.content, end="", flush=True)
            final_content += msg.content

    print()
    return final_content


if __name__ == "__main__":
    chat_with_tools('hai')
    # chat_with_tools("What's the weather like in Tokyo?")
    # chat_with_tools("What is 1234 * 5678?")
    # chat_with_tools("What's the weather in Paris in fahrenheit, and also calculate 99 * 42")


User: hai

Assistant: The user just said "hai" which appears to be a casual greeting (likely meant to be "hi"). This is a simple greeting that doesn't require any of the available tools (get_weather or calculate). I should respond with a friendly greeting back.Hello! 👋 How's your day going? Is there anything I can help you with today?


In [29]:
import ollama
import json

MODEL = "qwen3.5"

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country, e.g. 'London, UK'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical calculation",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '2 + 2'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# ANSI color codes
GRAY    = "\033[90m"
YELLOW  = "\033[93m"
CYAN    = "\033[96m"
GREEN   = "\033[92m"
MAGENTA = "\033[95m"
RESET   = "\033[0m"
DIM     = "\033[2m"
BOLD    = "\033[1m"


def print_thinking_chunk(text):
    """Print thinking tokens dimmed and in gray."""
    print(f"{DIM}{GRAY}{text}{RESET}", end="", flush=True)


def stream_response(messages, label="Assistant"):
    """Stream a single ollama.chat call, handling thinking + content + tool_calls."""
    collected_content = ""
    collected_tool_calls = []
    in_thinking = False

    stream = ollama.chat(
        model=MODEL,
        messages=messages,
        tools=tools,
        stream=True,
        think=False # Cancel thinking model if exist in model
    )

    for chunk in stream:
        msg = chunk.message

        # --- Thinking ---
        if msg.thinking:
            if not in_thinking:
                # Print thinking header once
                print(f"\n{DIM}{GRAY}┌─ Thinking {'─'*35}{RESET}")
                print(f"{DIM}{GRAY}│ {RESET}", end="", flush=True)
                in_thinking = True
            # Replace newlines to keep the block indented
            formatted = msg.thinking.replace("\n", f"\n{DIM}{GRAY}│ {RESET}")
            print(f"{DIM}{GRAY}{formatted}{RESET}", end="", flush=True)

        # --- Content ---
        if msg.content:
            if in_thinking:
                # Close thinking block before printing content
                print(f"\n{DIM}{GRAY}└{'─'*42}{RESET}")
                print(f"\n{CYAN}{BOLD}{label}:{RESET} ", end="", flush=True)
                in_thinking = False
            print(f"{CYAN}{msg.content}{RESET}", end="", flush=True)
            collected_content += msg.content

        # --- Tool calls ---
        if msg.tool_calls:
            for tc in msg.tool_calls:
                collected_tool_calls.append(tc)

    # Close thinking block if no content followed
    if in_thinking:
        print(f"\n{DIM}{GRAY}└{'─'*42}{RESET}")

    print()  # newline
    return collected_content, collected_tool_calls


def execute_tool(name, args):
    if name == "get_weather":
        location = args.get("location", "Unknown")
        unit = args.get("unit", "celsius")
        return {
            "location": location,
            "temperature": 22 if unit == "celsius" else 71,
            "unit": unit,
            "condition": "Sunny",
            "humidity": "60%"
        }
    elif name == "calculate":
        expr = args.get("expression", "")
        try:
            return {"expression": expr, "result": eval(expr)}
        except Exception as e:
            return {"error": str(e)}
    return {"error": "Unknown tool"}


def chat_with_tools(user_message):
    print(f"\n{'═'*50}")
    print(f"{BOLD}{YELLOW}User:{RESET} {user_message}")
    print(f"{'═'*50}")

    messages = [{"role": "user", "content": user_message}]

    # --- Round 1 ---
    print(f"\n{CYAN}{BOLD}Assistant:{RESET} ", end="", flush=True)
    collected_content, collected_tool_calls = stream_response(messages)

    if not collected_tool_calls:
        return None

    # --- Round 2: Execute tools ---
    messages.append({
        "role": "assistant",
        "content": collected_content or "",
        "tool_calls": [
            {
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            }
            for tc in collected_tool_calls
        ]
    })

    for tc in collected_tool_calls:
        name = tc.function.name
        args = tc.function.arguments
        result = execute_tool(name, args)

        print(f"\n{MAGENTA}⚙ Tool:{RESET} {BOLD}{name}{RESET}({json.dumps(args)})")
        print(f"{MAGENTA}  Result:{RESET} {json.dumps(result)}")

        messages.append({
            "role": "tool",
            "content": json.dumps(result)
        })

    # --- Round 3: Final response ---
    print(f"\n{CYAN}{BOLD}Assistant:{RESET} ", end="", flush=True)
    final_content, _ = stream_response(messages)
    return final_content


if __name__ == "__main__":
    # chat_with_tools("hai")
    # chat_with_tools("What's the weather like in Tokyo?")
    chat_with_tools("What is 1234 * 5678?")
    # chat_with_tools("What's the weather in Paris in fahrenheit, and also calculate 99 * 42")


══════════════════════════════════════════════════
User: What is 1234 * 5678?
══════════════════════════════════════════════════

Assistant: 

⚙ Tool: calculate({"expression": "1234 * 5678"})
  Result: {"expression": "1234 * 5678", "result": 7006652}

Assistant: 1234 multiplied by 5678 is 7,006,652.


In [47]:
from langchain_ollama import ChatOllama

In [62]:
llm = ChatOllama(model="qwen3.5", reasoning=True)

In [13]:
# messages = [
#     (
#         "system",
#         "You are a helpful assistant that translates English to French. Translate the user sentence.",
#     ),
#     ("human", "I love programming."),
# ]
# ai_msg = llm.invoke(messages)
# ai_msg

KeyboardInterrupt: 

In [67]:
stream_out = llm.stream('hai, how are you', think=False)

In [68]:
thinking = False
for i in stream_out:
    if i.additional_kwargs.get('reasoning_content', ''):
        if not thinking:
            print('Model Thinking')
            thinking = True
        
        print(i.additional_kwargs['reasoning_content'], end="")
    
    if i.content:
        if thinking:
            thinking = False
            print()
            print('Model: ')
        
        print(i.content, end='')


Hello! I'm doing well, thanks for asking. How about you? Is there anything you'd like to chat about or need help with today?

In [54]:
from typing import TypedDict

class TranslationDict(TypedDict):
    email_content: str
    sender: str
    reciever: str
    title: str


struc_llm = llm.with_structured_output(TranslationDict)

In [56]:
struc_llm.invoke("sent a mail for my mom, telling thanks for beign always with me, you can decide everything else, i am vajid. dont think about anything just give me the mail content, nothing else")

{'email_content': "Subject: Thank You\n\nDear Mom,\n\nI wanted to take a moment to tell you how much I appreciate you for always being with me. Your support means the world to me.\n\nRegarding everything else, please know that you can decide. I trust your judgment and just want to thank you for being there.\n\nPlease don't think about anything heavy or worry too much.\n\nWith all my love,\n\nVajid",
 'sender': 'Vajid',
 'reciever': 'Mom',
 'title': 'Thank You'}